# Name That LLM: DeBERTa-v3-large, trained once on the whole clean pool

**Goal:** P8's first new member (spec §5.3). The pool's maintainer took DeBERTa-v3-large to
0.587 public LB alone, trained on 193k clean passages; our best ensemble scores 0.454.

**One honest run instead of five folds** (`pool_member.py`):
- it never trains on one of our 2,699 passages;
- it never trains on a pool passage about one of our 150 prompts.

So its predictions on ALL our passages are honest, like the hidden test's, from one run.
Every label weighs the same in the loss, since the pool's sources cover the 18 models very
unevenly.

**This session:**
1. a speed probe on GPU 0 sizes the run to the session;
2. two seeds train in parallel, one per T4, each on its own sample of the pool;
3. their average is the member.

The gate (`scripts/ensemble_gate.py`) then runs on the PC.

Private until 2027-01-13 (P8, spec §3). Part of
[github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P8 (Task 4).

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
%pip install -q "name-that-llm @ git+https://github.com/Ryqn0/name-that-llm@37f4dd52cb39ede587647040facd6321835f99eb"

In [ ]:
import json
import os
import subprocess
import sys
import time
from contextlib import ExitStack
from pathlib import Path

import numpy as np
import torch

from name_that_llm.config import EncoderConfig, load_config
from name_that_llm.data import LABELS, load_train
from name_that_llm.metrics import score
from name_that_llm.pool import build_pool, load_pool_sources, source_counts
from name_that_llm.pool_member import honest_training_set, rows_for_budget

START = time.time()
# Kaggle's folders by default. The smoke test (tests/test_notebooks.py) runs this very code
# on a CPU, with these pointed at a tiny DeBERTa-style model and a fake pinned pool.
INPUT = Path(os.environ.get("NTL_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("NTL_WORK", "/kaggle/working"))
CONFIG = os.environ.get("NTL_ENCODER_CONFIG", "deberta_pool")  # a name, or a YAML path
ON_KAGGLE = "NTL_INPUT" not in os.environ
# Kaggle stops a session at 12 hours, or when the weekly GPU quota runs out, and a stopped
# run can lose its outputs. Set from `kaggle quota` before each push (the plan's rule).
SESSION_MIN = 10.5 * 60
SEEDS = (42, 43)  # one per T4

GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS)
if ON_KAGGLE:
    assert len(GPUS) == 2, "this notebook needs the T4 x2 accelerator"
train = load_train()
y = train["model"].to_numpy()
labels = np.asarray(LABELS)
cfg = load_config(CONFIG, model_cls=EncoderConfig)
print(cfg)


def elapsed_minutes() -> float:
    return (time.time() - START) / 60


def on_gpu(gpu: int) -> dict:
    return {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONIOENCODING": "utf-8"}


def member_cmd(*args: str) -> list[str]:
    return [sys.executable, "-m", "name_that_llm.pool_member", "--config", CONFIG, *args]

## 1. The pinned pool

Our private copy of the clean pool v10 (`ryqn00/name-that-llm-pool-v10`), every source
kept except any the maintainer dropped since (`configs/pool_all.yaml`). The same safeguards
as always (`pool.build_pool`) run on it. Its per-source counts are saved, so a change shows
at once. Counts only are printed, never a passage.

In [ ]:
pool = build_pool(load_pool_sources("pool_all"), INPUT, train)
POOL_PATH = WORK / "pool.parquet"
pool.to_parquet(POOL_PATH)
counts = source_counts(pool)
(WORK / "pool_counts.json").write_text(json.dumps(counts, indent=1), encoding="utf-8")
honest = honest_training_set(pool, train, None, cfg.seed)
print(f"pool: {len(pool):,} passages from {len(counts)} sources; honest: {len(honest):,}")

## 2. The speed probe, then the run's size

In a process of its own, as the real run would train: one step on the worst batch (the
longest passages), then one epoch on 256 passages. Version 1 ran out of memory 8 minutes
into training, because its probe never met a batch of long passages. So now the runs don't
start if the worst batch takes more than 85% of the GPU. If float16 overflowed (DeBERTa-v3
can, on T4s), the probe timed float32 too, and the runs use it. The size is then whatever
fits the session, with 20 minutes kept for predicting.

In [ ]:
probe_path = WORK / "probe.json"
subprocess.run(
    member_cmd("--pool", str(POOL_PATH), "--probe", "256", "--out", str(probe_path)),
    env=on_gpu(0),
    check=True,
)
probe = json.loads(probe_path.read_text(encoding="utf-8"))
FP32 = not probe["finite"]  # the probe then timed float32: seconds_per_passage is its speed
# 85%: the CUDA context sits outside PyTorch's count, and a long run's memory fragments.
if probe["gpu_memory_gb"] and probe["peak_memory_gb"] > 0.85 * probe["gpu_memory_gb"]:
    raise SystemExit(
        f"the worst batch takes {probe['peak_memory_gb']} of the GPU's "
        f"{probe['gpu_memory_gb']} GB: lower batch_size"
    )
minutes = SESSION_MIN - elapsed_minutes() - 20
ROWS = rows_for_budget(probe["seconds_per_passage"], cfg.epochs, minutes, len(honest))
print(probe, f"| float32: {FP32} | {minutes:.0f} min left -> {ROWS:,} passages per seed")
if ROWS == 0:
    raise SystemExit("no time left for a run inside the session")

## 3. Two seeds, one per T4

Each process trains a fresh model on its own sample of `ROWS` honest passages (seeded),
saves each finished epoch, and predicts all our passages. If one fails, the other is
stopped at once.

In [ ]:
run_start, procs = time.time(), []
(WORK / "runs").mkdir(exist_ok=True)
with ExitStack() as logs:  # every log is closed, however the wait ends
    for gpu, seed in enumerate(SEEDS):
        args = ["--pool", str(POOL_PATH), "--seed", str(seed), "--rows", str(ROWS)]
        args += ["--out", str(WORK / "runs" / f"seed{seed}.npz")]
        args += ["--save-dir", str(WORK / "models" / f"seed{seed}")]
        if FP32:
            args.append("--fp32")
        log = logs.enter_context(open(WORK / "runs" / f"seed{seed}.log", "w"))
        procs.append(
            subprocess.Popen(
                member_cmd(*args), env=on_gpu(gpu), stdout=log, stderr=subprocess.STDOUT
            )
        )
    while None in (codes := [p.poll() for p in procs]):
        if any(code not in (None, 0) for code in codes):
            break  # one failed: stop the other below, rather than wait for it
        time.sleep(15)
    for p in procs:
        if p.poll() is None:
            p.kill()
            p.wait()
# From the codes polled BEFORE the kill: a seed stopped because the other failed isn't the
# cause (version 1 blamed seed 42, which it had stopped itself).
failed = [s for s, code in zip(SEEDS, codes, strict=True) if code not in (None, 0)]
if failed:
    tail = (WORK / "runs" / f"seed{failed[0]}.log").read_text(encoding="utf-8", errors="replace")
    print(tail[-3000:])
    raise RuntimeError(f"seed {failed[0]} failed")
print(f"both seeds trained in {(time.time() - run_start) / 60:.0f} min")

## 4. The member: the two seeds' average

Balanced accuracy on all our passages, each seed alone and averaged. `mean.npy` is the
member's predictions, for the gate on the PC (`scripts/ensemble_gate.py --extra
deberta=mean.npy`).

In [ ]:
probas = {s: np.load(WORK / "runs" / f"seed{s}.npz")["train"] for s in SEEDS}
mean = np.mean(list(probas.values()), axis=0)
np.save(WORK / "mean.npy", mean)
scores = {str(s): score(y, labels[p.argmax(axis=1)]) for s, p in probas.items()}
scores["mean"] = score(y, labels[mean.argmax(axis=1)])
summary = {"rows": ROWS, "epochs": cfg.epochs, "fp32": FP32, "scores": scores}
summary["minutes"] = round(elapsed_minutes(), 1)
(WORK / "results.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")
print(json.dumps(summary, indent=1))

## Findings

_Filled in after the run._